<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 03. Transformer

## Attention Is All You Need

- **원 논문:** [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- **저자 / 발표:** Ashish Vaswani et al. · NeurIPS 2017 (2017)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** scaled dot-product, multi-head 분할, sinusoidal position을 직접 재현한다.

**축소하거나 생략한 부분:** 6-layer encoder-decoder, WMT 대신 1 encoder block과 sentiment 분류를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.2.1 Eq. (1): softmax(QKᵀ/√dk)V | Task 1–2 | row sum/mask |
| §3.2.2: Multi-Head Attention | Task 2 | 4 heads |
| §3.4: embeddings | Task 2 | shared sequence tensors |
| §3.5 Eq. (3): positional encoding | Task 1 | sin/cos values |

## 핵심 아이디어와 수식

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

**기호 해설:** Q/K/V는 query/key/value, d_k는 head 차원, softmax는 key 축 정규화다.

**코드 대응:** Task 1이 attention/position 함수, Task 2가 MultiHeadAttention과 encoder를 구현한다.

**입력과 출력 shape:** token [B,4] → embedding [B,4,24] → 4 heads [B,4,4,6] → logits [B,2].

**포트폴리오 구현 범위:** decoder/cross-attention은 제외하고 논문의 encoder self-attention 핵심을 직접 구현한다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.2.1 Eq. (1): softmax(QKᵀ/√dk)V
- **노트북에서 구현할 부분:** Task 1–2
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** row sum/mask

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
@dataclass
class TransformerConfig:
    model_dim: int = 24
    heads: int = 4
    feedforward_dim: int = 48
    num_classes: int = 2
    learning_rate: float = 0.018
    steps: int = 18


def scaled_dot_product_attention(query, key, value, mask=None):
    scale = math.sqrt(query.shape[-1])
    scores = torch.matmul(query, key.transpose(-2, -1)) / scale
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = scores.softmax(dim=-1)
    return torch.matmul(weights, value), weights


def sinusoidal_positions(length, dimension, device):
    positions = torch.arange(length, device=device).float()[:, None]
    even_indices = torch.arange(0, dimension, 2, device=device).float()
    frequencies = torch.exp(-math.log(10000.0) * even_indices / dimension)
    encoding = torch.zeros(length, dimension, device=device)
    encoding[:, 0::2] = torch.sin(positions * frequencies)
    encoding[:, 1::2] = torch.cos(positions * frequencies)
    return encoding


config = TransformerConfig()
position = sinusoidal_positions(4, config.model_dim, DEVICE)
assert position.shape == (4, 24)
assert torch.allclose(position[0, 0::2], position.new_zeros(12))

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.2.2: Multi-Head Attention
- **노트북에서 구현할 부분:** Task 2
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** 4 heads

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = config.heads
        self.head_dim = config.model_dim // config.heads
        self.query = nn.Linear(config.model_dim, config.model_dim)
        self.key = nn.Linear(config.model_dim, config.model_dim)
        self.value = nn.Linear(config.model_dim, config.model_dim)
        self.output = nn.Linear(config.model_dim, config.model_dim)

    def split_heads(self, tensor):
        batch, length, _ = tensor.shape
        tensor = tensor.view(batch, length, self.heads, self.head_dim)
        return tensor.transpose(1, 2)

    def merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.view(batch, length, self.heads * self.head_dim)

    def forward(self, inputs, mask=None):
        query = self.split_heads(self.query(inputs))
        key = self.split_heads(self.key(inputs))
        value = self.split_heads(self.value(inputs))
        attended, weights = scaled_dot_product_attention(
            query,
            key,
            value,
            mask,
        )
        return self.output(self.merge_heads(attended)), weights


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.model_dim)
        self.attention = MultiHeadSelfAttention(config)
        self.norm1 = nn.LayerNorm(config.model_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(config.model_dim, config.feedforward_dim),
            nn.ReLU(),
            nn.Linear(config.feedforward_dim, config.model_dim),
        )
        self.norm2 = nn.LayerNorm(config.model_dim)
        self.classifier = nn.Linear(config.model_dim, config.num_classes)

    def encode(self, token_ids):
        embedded = self.embedding(token_ids)
        embedded = embedded + sinusoidal_positions(
            token_ids.shape[1],
            embedded.shape[-1],
            token_ids.device,
        )
        attended, weights = self.attention(embedded)
        hidden = self.norm1(embedded + attended)
        hidden = self.norm2(hidden + self.feedforward(hidden))
        return hidden, weights

    def forward(self, token_ids):
        hidden, _ = self.encode(token_ids)
        return self.classifier(hidden.mean(dim=1))


model = TransformerClassifier(VOCAB_SIZE, config).to(DEVICE)
hidden, attention_weights = model.encode(sentence_ids[:2])
assert hidden.shape == (2, 4, 24)
assert attention_weights.shape == (2, 4, 4, 4)
assert model(sentence_ids[:2]).shape == (2, 2)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.5 Eq. (3): positional encoding
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** sin/cos values

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def transformer_loss(logits, targets):
    return F.cross_entropy(logits, targets)


def train_transformer_step(model, optimizer, inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = transformer_loss(model(inputs), targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_transformer(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        predictions = model(inputs).argmax(dim=1)
    return float((predictions == targets).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    losses.append(
        train_transformer_step(
            model,
            optimizer,
            sentence_ids[:240],
            sentiment_labels[:240],
        )
    )
accuracy = evaluate_transformer(
    model,
    sentence_ids[240:],
    sentiment_labels[240:],
)
assert min(losses[3:]) < losses[0]
assert 0.0 <= accuracy <= 1.0
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.title(f"Transformer accuracy={accuracy:.1%}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 6-layer encoder-decoder, WMT 대신 1 encoder block과 sentiment 분류를 사용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.